# Insurance Claim Prediction
## 04. Modeling & Evaluation

This notebook focuses on building, training and evaluating machine learning models for insurance claim prediction.

The objective is to develop a predictive model that can estimate the likelihood of an insurance claim based on property and policy-related features.

## 1. Import Required Libraries

In this section, we import all the essential libraries needed for data manipulation, model building, evaluation, and model persistence.


In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

## 2.📂 Load Dataset

The processed dataset is loaded for model training and evaluation.  
This dataset has already undergone preprocessing and feature engineering.

In [2]:

X_processed_df = pd.read_csv("X_processed.csv")
y = pd.read_csv("y.csv")

print(X_processed_df.shape)
print(y.shape)


(7160, 1335)
(7160, 1)


## 3. Train-Test Split

The dataset is split into training and testing sets to evaluate the model’s ability to generalize to unseen data.

- **Training set:** Used for model learning
- **Testing set:** Used for performance evaluation

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed_df, y, test_size=0.2, random_state=42
)

## 4. Model Training

In this section, we train a machine learning classifier to predict insurance claim outcomes.

The model learns the relationship between input features and the target variable.

## 1. Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [5]:
# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)

log_reg.fit(X_train, y_train.values.ravel())

y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.7807262569832403
              precision    recall  f1-score   support

           0       0.79      0.97      0.87      1098
           1       0.61      0.16      0.26       334

    accuracy                           0.78      1432
   macro avg       0.70      0.57      0.56      1432
weighted avg       0.75      0.78      0.73      1432



## Logistic Regression Model Insight

A Logistic Regression model was trained as the baseline classifier for the insurance claim prediction task.

This model serves as a simple and interpretable benchmark to evaluate the predictive performance of more advanced models.

### Model Performance
- **Accuracy:** 78.1%
- **Weighted F1-score:** 0.73
- **Recall for No Claim (Class 0):** 97%
- **Recall for Claim (Class 1):** 16%

### Interpretation
The model achieved a strong overall accuracy of **78.1%**, indicating good general predictive performance.

However, a closer look at the classification report shows that the model performs significantly better in predicting **non-claim cases (Class 0)** than **claim cases (Class 1)**.

The low recall of **16% for Class 1** suggests that the model misses many actual insurance claims.

This may be due to:
- class imbalance in the dataset
- overlapping feature distributions
- insufficient signal in the predictors

This baseline result motivates the use of:
- **hyperparameter tuning**
- **ensemble models such as Random Forest**
- **class balancing techniques**

to improve performance on the minority class.

## 2. Tuned Logistic Regression

In [9]:
from sklearn.model_selection import GridSearchCV

In [8]:
# Hyperparameter tuning
param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"],
    "penalty": ["l2"]
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train.values.ravel())

best_lr = grid_lr.best_estimator_

y_pred_best_lr = best_lr.predict(X_test)

print("Best Parameters:", grid_lr.best_params_)
print("Tuned Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_best_lr))
print(classification_report(y_test, y_pred_best_lr))

Best Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Tuned Logistic Regression Accuracy: 0.7828212290502793
              precision    recall  f1-score   support

           0       0.79      0.98      0.87      1098
           1       0.68      0.13      0.22       334

    accuracy                           0.78      1432
   macro avg       0.73      0.56      0.55      1432
weighted avg       0.76      0.78      0.72      1432



## Tuned Logistic Regression Model Insight

To improve the baseline model performance, hyperparameter tuning was performed using **GridSearchCV** with 5-fold cross-validation.

The following hyperparameters were optimized:

- **C:** [0.01, 0.1, 1, 10]
- **Penalty:** l2
- **Solver:** liblinear, lbfgs

### Best Parameters
- **C = 0.1**
- **Penalty = l2**
- **Solver = lbfgs**

### Model Performance
- **Accuracy:** 78.28%
- **Weighted F1-score:** 0.72
- **Recall for No Claim (Class 0):** 98%
- **Recall for Claim (Class 1):** 13%

### Interpretation
The tuned logistic regression model produced a **slight improvement in overall accuracy** compared to the baseline logistic regression model.

However, the recall for the positive claim class remains low, indicating that the model still struggles to correctly identify actual insurance claims.

This suggests that while hyperparameter tuning improved generalization slightly, the linear nature of logistic regression may still limit performance on this problem.

This result motivates the evaluation of a more flexible ensemble model such as **Random Forest**, which may better capture nonlinear relationships in the data.

## 3. Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

In [11]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train.values.ravel())

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.7646648044692738
              precision    recall  f1-score   support

           0       0.80      0.92      0.86      1098
           1       0.49      0.25      0.33       334

    accuracy                           0.76      1432
   macro avg       0.65      0.58      0.59      1432
weighted avg       0.73      0.76      0.73      1432



## Random Forest Model Insight

A Random Forest classifier was trained to capture more complex and nonlinear relationships within the insurance claim dataset.

Unlike Logistic Regression, Random Forest can model interactions between variables and is often more robust for structured tabular data.

### Model Performance
- **Accuracy:** 76.47%
- **Weighted F1-score:** 0.73
- **Recall for No Claim (Class 0):** 92%
- **Recall for Claim (Class 1):** 25%

### Interpretation
Although the overall accuracy is slightly lower than the Logistic Regression models, the Random Forest model achieved a **higher recall for claim cases (Class 1)**.

This indicates improved ability to identify actual insurance claims, which is often more valuable in a business setting where missing true claims can be costly.

The improvement in minority class detection suggests that Random Forest better captures nonlinear patterns and feature interactions within the data.

## 5. Model Comparison and Final Selection

The performance of all trained models is summarized below:

| Model | Accuracy | Claim Recall |
|--------|----------|-------------|
| Logistic Regression | 78.07% | 16% |
| Tuned Logistic Regression | 78.28% | 13% |
| Random Forest | 76.47% | 25% |

### Final Model Selection
While Tuned Logistic Regression achieved the highest overall accuracy, the Random Forest model was selected as the preferred model for deployment.

This decision was based on its stronger ability to detect actual insurance claim cases, as reflected by the higher recall for Class 1.

In real-world insurance applications, correctly identifying claim cases is often more important than maximizing overall accuracy.

Therefore, Random Forest provides a better balance between predictive performance and business relevance.

## 6. Project Conclusion

This project successfully developed a machine learning solution for predicting the probability of insurance claims based on building-related features.

The workflow covered the complete data science lifecycle, including:

- data understanding
- exploratory data analysis
- preprocessing and feature engineering
- model training and evaluation
- model selection and deployment

Three models were evaluated: **Logistic Regression**, **Tuned Logistic Regression**, and **Random Forest**.

Although Logistic Regression achieved slightly higher overall accuracy, the **Random Forest model was selected as the final model** because it demonstrated better performance in identifying actual insurance claim cases, with a higher recall for the positive class.

This makes the model more suitable for real-world insurance risk assessment, where detecting true claim cases is more critical than maximizing accuracy alone.

The final model was deployed as an interactive **Streamlit web application**, enabling users to input building information and obtain real-time insurance claim predictions.

This project demonstrates practical skills in:
- machine learning model development
- model evaluation and selection
- business-focused decision making
- model deployment using Streamlit
- version control with GitHub

Overall, the solution provides a scalable and user-friendly predictive system that can support risk management and decision-making in the insurance domain.

In [12]:
import joblib
joblib.dump(rf_model, "model.pkl")

['model.pkl']